# 02 - Operações DML em Delta Lake

Este notebook executa `INSERT`, `UPDATE` e `DELETE` em uma tabela Delta armazenada no bucket `bronze`.

In [ ]:
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.554 pyspark-shell"
)

builder = SparkSession.builder.appName("dml-delta")
builder = builder.config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
builder = builder.config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
builder = builder.config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
builder = builder.config("spark.hadoop.fs.s3a.access.key", "minioadmin")
builder = builder.config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
builder = builder.config("spark.hadoop.fs.s3a.path.style.access", "true")
builder = builder.config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [ ]:
base_path = "s3a://bronze/funcionarios"
delta_table = DeltaTable.forPath(spark, base_path)
print("Estado inicial da tabela funcionarios:")
spark.read.format("delta").load(base_path).show(10, False)

In [ ]:
print("Inserindo novo funcionário...")
novos = spark.createDataFrame([
    (4, "Lina", "Analista de Testes", 4200.0, 1),
], ["id", "nome", "cargo", "salario", "departamento_id"])
novos.write.format("delta").mode("append").save(base_path)

In [ ]:
print("Atualizando salário do funcionário id = 2...")
delta_table = DeltaTable.forPath(spark, base_path)
delta_table.update(
    condition="id = 2",
    set={"salario": "salario * 1.10"},
)

In [ ]:
print("Removendo funcionário com id = 3...")
delta_table.delete(
    condition="id = 3"
)

In [ ]:
print("Estado final da tabela funcionarios:")
spark.read.format("delta").load(base_path).show(10, False)

In [ ]:
print("Histórico de transações:")
DeltaTable.forPath(spark, base_path).history().select("version", "timestamp", "operation").show(False)

In [ ]:
print("Leitura versão 0 (time travel):")
spark.read.format("delta").option("versionAsOf", 0).load(base_path).show(10, False)
spark.stop()